# Exception Handling

## Introduction

Ref: https://docs.julialang.org/en/v1/manual/control-flow/#Exception-Handling

Like other languages, Julia has a bunch of [builtin exceptions](https://docs.julialang.org/en/v1/manual/control-flow/#Built-in-Exceptions). I can handle thrown exceptions from other functions, throw exceptions in my own functions, and define custom exceptions should I so choose. The style guide recommends against overusing try/catch blocks.

## Throwing Exceptions

Exceptions can be thrown in two different ways - 

  * Use the `throw()` function.
  * Use the `error()` function.

With the `throw()` function I can instantiate any exception type and throw it. The `error()` function is more of a convenience that throws the builtin `ErrorException` object with a user specified error message. It is good to use in quick scripts, but for libraries I should stick to `throw()`.

According to the manual, when writing error message, it is preferred to make the first word lowercase.

In [1]:
function addpos(x::Integer, y::Integer)
    if x < 0 || y < 0
        throw(ArgumentError("both $x and $y must be positive!"))
    end
    x + y
end

addpos (generic function with 1 method)

In [2]:
addpos(-1, 2)

ArgumentError: ArgumentError: both -1 and 2 must be positive!

In [3]:
addpos(2, -1)

ArgumentError: ArgumentError: both 2 and -1 must be positive!

In [4]:
addpos(1, 2)

3

In [5]:
function mulpos(x::Integer, y::Integer)
    if x < 0 || y < 0
        error("both $x and $y must be positive!")
    end
    x * y
end

mulpos (generic function with 1 method)

In [6]:
mulpos(-1, 2)

ErrorException: both -1 and 2 must be positive!

## Custom Exceptions

In [7]:
struct MyCustomException <: Exception
    errcode::Integer
    msg::String
end

In [8]:
function dostuff()
    status = rand()
    if status < 0.25
        throw(MyCustomException(400, "client has done something wrong"))
    elseif status < 0.5
        throw(MyCustomException(500, "server is broken"))
    else
        42
    end
end

dostuff (generic function with 1 method)

In [9]:
dostuff()

MyCustomException: MyCustomException(400, "client has done something wrong")

In [10]:
dostuff()

42

In [11]:
dostuff()

MyCustomException: MyCustomException(500, "server is broken")

## Catching Exceptions 

Julia does not have multiple catch blocks, one per each type of exception that I want to handle. There is just one catch block that captures the exception object and then I have to reflect on the type. In addition to `try` and `catch`, there is also the `else` block for doing things in case there are no errors in the `try` block, and a `finally` block do stuff regardless of whether an error happend.

```julia
try
    risky code here
catch err
    exception handling here
else
    in case of no error
finally
    regardless of whatever happens
end
```

`else` and `finally` are optional.

In [12]:
struct ClientError <: Exception end

struct ServerError <: Exception end

function get()
    status = rand()
    if status < 0.25
        throw(ClientError())
    elseif status < 0.5
        throw(ServerError())
    else
        42
    end
end

function do_get()
    x = 0
    try
        x = get()
    catch err
        if err isa ClientError
            println("Got client error, retry.")
        elseif err isa ServerError
            println("Got server error, wait")
        else
            println("something else went wrong")
        end
    else
        println("Now do something else with $x")
    finally
        println("Cleanup on aisle 42.")
    end
end

do_get (generic function with 1 method)

In [13]:
do_get()

Got server error, wait
Cleanup on aisle 42.


In [14]:
do_get()

Now do something else with 42
Cleanup on aisle 42.


In [24]:
do_get()

Got client error, retry.
Cleanup on aisle 42.


In [30]:
struct YourFault <: Exception end

function do_get_2()
    x = 0
    try
        x = get()
    catch err
        if err isa ClientError
            rethrow(YourFault())
        else
            rethrow()
        end
    end
end

do_get_2 (generic function with 1 method)

In [31]:
do_get_2()

42

In [32]:
do_get_2()

YourFault: YourFault()

In [40]:
do_get_2()

ServerError: ServerError()